# Figure 4: Spatial Maps of ET Component Differences

Data source: 5-year averaged (2010-2014) CLM5 output from `G:/Hangkai/CTH_ET project/Final_data/`

In [ ]:
import netCDF4
import os
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Load all data from 5-year averaged Final_data
# ============================================================
file_path = r'G:\Hangkai\CTH_ET project\Final_data'

scenarios_EX = ['CLM Default', 'GEDI Max', 'GEDI Mean', 'GEDI Median']

variable_names = [
    'FCEV', 'FCTR', 'QVEGE', 'QVEGT', 'FGEV', 'FSH', 'FSH_G', 'FSH_V', 'EFLX_LH_TOT',
    'EFLX_SOIL_GRND', 'FPSN', 'FPSN_WC', 'FPSN_WJ', 'FPSN_WP', 'Qh', 'Qle', 'Q2M',
    'QFLX_EVAP_TOT', 'QFLX_SNOW_GRND', 'QFLX_RAIN_GRND', 'Rnet', 'TV',
    'TSA', 'Z0HV', 'Z0M', 'Z0MV', 'Z0QV'
]

scenario_mapping_EX = {
    'CLM Default': 'CLM Default.nc',
    'GEDI Max': 'GEDI Max.nc',
    'GEDI Mean': 'GEDI Mean.nc',
    'GEDI Median': 'GEDI Median.nc',
}

# Load PFT structure from Final_data
with netCDF4.Dataset(os.path.join(file_path, 'CLM Default.nc'), 'r') as ds:
    pfts1d_itype_veg = np.array(ds.variables['pfts1d_itype_veg'][:])
    pfts1d_ixy       = np.array(ds.variables['pfts1d_ixy'][:])
    pfts1d_jxy       = np.array(ds.variables['pfts1d_jxy'][:])
    pfts1d_wtgcell   = np.array(ds.variables['pfts1d_wtgcell'][:])

# Handle potential time dimension
if pfts1d_itype_veg.ndim > 1:
    pfts1d_itype_veg = pfts1d_itype_veg[0, :]
    pfts1d_ixy = pfts1d_ixy[0, :]
    pfts1d_jxy = pfts1d_jxy[0, :]
    pfts1d_wtgcell = pfts1d_wtgcell[0, :]

# Load lat/lon from CLM input (needed for 2D mapping)
file_path_input = r'H:\CLM_input'
with netCDF4.Dataset(os.path.join(file_path_input, 'mean.nc'), 'r') as ds:
    lat2d = ds.variables['LATIXY'][:]
    lon2d = ds.variables['LONGXY'][:]
lon2d = lon2d - 180

# PFT names for tree PFTs
pft_names = {
    "NET Temperate": 1, "NET Boreal": 2, "NDT Boreal": 3,
    "BET Tropical": 4, "BET Temperate": 5,
    "BDT Tropical": 6, "BDT Temperate": 7, "BDT Boreal": 8,
}
unique_pfts = pft_names.keys()

grid_shape = (len(lat2d), len(lon2d[0]))

# Load ET data for all scenarios
data_EX = {}
for scenario in scenario_mapping_EX:
    data_EX[scenario] = {}
    nc_file = os.path.join(file_path, scenario_mapping_EX[scenario])
    with netCDF4.Dataset(nc_file, 'r') as ds:
        for variable in variable_names:
            data_EX[scenario][variable] = ds.variables[variable][:]

# Compute 2D gridded annual means
annual_data_EX = {var: {} for var in variable_names}

for scenario in scenario_mapping_EX:
    for variable in variable_names:
        variable_2d_grid = np.zeros(grid_shape)
        for pft_name in unique_pfts:
            pft = pft_names[pft_name]
            pft_indexes = np.where(pfts1d_itype_veg == pft)[0]
            variable_values = np.mean(data_EX[scenario][variable], axis=0) * pfts1d_wtgcell
            variable_pft = variable_values[pft_indexes]

            for i, index in enumerate(pft_indexes):
                row = pfts1d_jxy[index] - 1
                col = pfts1d_ixy[index] - 1
                variable_2d_grid[row, col] += variable_pft[i]

        annual_data_EX[variable][scenario] = variable_2d_grid

# Define comparison combinations
combinations = [
    ('GEDI Mean', 'CLM Default'),
    ('GEDI Max', 'CLM Default'),
    ('GEDI Max', 'GEDI Mean'),
    ('GEDI Median', 'GEDI Mean')
]

variables_to_compare = ['FCEV', 'FCTR', 'FGEV']

# Calculate ET
for scenario in data_EX:
    data_EX[scenario]['ET'] = (data_EX[scenario]['FCEV'] +
                                data_EX[scenario]['FCTR'] +
                                data_EX[scenario]['FGEV'])

# Compute percentage differences
percentage_differences = {}
for combination in combinations:
    scenario1, scenario2 = combination
    percentage_differences[combination] = {}
    for variable in variables_to_compare:
        diff = ((data_EX[scenario1][variable] - data_EX[scenario2][variable])
                / data_EX[scenario2][variable] * 100)
        percentage_differences[combination][variable] = diff
    diff_ET = ((data_EX[scenario1]['ET'] - data_EX[scenario2]['ET'])
               / data_EX[scenario2]['ET'] * 100)
    percentage_differences[combination]['ET'] = diff_ET

print('Data loaded and processed successfully.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.colors import ListedColormap

# ============================================================
# Figure 4: Spatial maps of ET component differences
# ============================================================
projection = ccrs.PlateCarree(central_longitude=180)

variables = ['ET', 'FCEV', 'FCTR', 'FGEV']

combinations = [
    ('GEDI Mean', 'CLM Default'),
    ('GEDI Max', 'CLM Default'),
    ('GEDI Max', 'GEDI Mean'),
    ('GEDI Median', 'GEDI Mean')
]

# Custom colormap with white center
turbo_cmap = plt.get_cmap('coolwarm')
colors = turbo_cmap(np.linspace(0, 1, 200))
colors[95:105] = (1, 1, 1, 1)
white_center_cmap = ListedColormap(colors)

fig, axes = plt.subplots(len(variables), len(combinations), figsize=(22, 18),
                         subplot_kw=dict(projection=projection), dpi=300)

for j, variable in enumerate(variables):
    for i, (scenario1, scenario2) in enumerate(combinations):
        ax = axes[j, i]

        if variable == 'ET':
            data_fcev = annual_data_EX['FCEV'][scenario1]
            data_fctr = annual_data_EX['FCTR'][scenario1]
            data_fgev = annual_data_EX['FGEV'][scenario1]
            data_et = data_fcev + data_fctr + data_fgev
        else:
            data_et = annual_data_EX[variable][scenario1]

        if variable == 'ET':
            ylabel = 'ET'
        elif variable == 'FCEV':
            ylabel = 'Canopy EV'
        elif variable == 'FCTR':
            ylabel = 'Canopy TR'
        elif variable == 'FGEV':
            ylabel = 'Ground EV'

        title = f'{scenario1} vs {scenario2}'

        if variable == 'ET':
            base_fcev = annual_data_EX['FCEV'][scenario2]
            base_fctr = annual_data_EX['FCTR'][scenario2]
            base_fgev = annual_data_EX['FGEV'][scenario2]
            base_data = base_fcev + base_fctr + base_fgev
        else:
            base_data = annual_data_EX[variable][scenario2]
        base_data[base_data == 0] = np.nan
        plot_data = (data_et - base_data) / base_data * 100
        plot_data = np.ma.masked_invalid(plot_data)
        levels = np.linspace(-10, 10, num=11)
        cmap = white_center_cmap

        cs = ax.contourf(lon2d, lat2d, plot_data, levels=levels, cmap=cmap,
                         extend='both', transform=projection)

        gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                          linewidth=0.5, color='gray', alpha=0.5,
                          xlocs=[-120, 0, 120], ylocs=[-90, -45, 0, 45, 90])
        labels = gl.xlabel_style
        labels['size'] = 0
        gl.xlabel_style = labels
        labels = gl.ylabel_style
        labels['size'] = 0
        gl.ylabel_style = labels
        gl.bottom_labels = False
        gl.right_labels = False

        ax.tick_params(labelsize=20)
        ax.coastlines()

        if j == 0:
            ax.set_title(title, fontsize=18)
            labels = gl.xlabel_style
            labels['size'] = 14
            gl.xlabel_style = labels

        if i == 0:
            ax.annotate(ylabel, xy=(-0.2, 0.5), xycoords='axes fraction',
                        fontsize=18, rotation=90, va='center')
            labels = gl.ylabel_style
            labels['size'] = 14
            gl.ylabel_style = labels

# Colorbar
cbar_ax = fig.add_axes([0.39, 0.22, 0.25, 0.015])
cbar = fig.colorbar(cs, cax=cbar_ax, orientation='horizontal', ticks=levels)
cbar.set_label("Relative Difference (%)", fontsize=18)
cbar.ax.tick_params(labelsize=14)
cbar.outline.set_visible(True)

plt.subplots_adjust(hspace=-0.67, wspace=0.08)
plt.savefig('figures/Figure_4_ET_Spatial_Maps.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/Figure_4_ET_Spatial_Maps.tiff', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')